# Financial News Summarizer

This project builds a **Financial News Summarizer** utilizing free news articles fetched through the [NewsData.io](https://newsdata.io) free API. The search keyword is focused on **Artificial Intelligence (AI)** related news.

Since this uses a free API key, there is a limit on the number of articles returned per request. The project is broken down into the following parts:

- **Part 1** — Data Mining from NewsData.io
- **Part 2** — Chunking
- **Part 3** — Embedding
- **Part 4** — Indexing with ChromaDB
- **Part 5** — Querying & Summarization

---

## Full Pipeline Summary

```
NewsData.io API          ← Free API, limited articles, keyword: "Artificial Intelligence"
      ↓
  Append_article         ← Extract title + description from each article
      ↓
  documents              ← Format each article as plain text
      ↓
  all_chunks             ← Split using RecursiveCharacterTextSplitter
      ↓                     chunk_size=300, chunk_overlap=50
  embeddings             ← Convert chunks to vectors via all-MiniLM-L6-v2 (local model)
      ↓
  ChromaDB               ← Persist indexed chunks to ./news_chroma_db
      ↓
  Query                  ← Embed user query using same local model
      ↓
  Top-K Chunks           ← Retrieve most relevant chunks by distance score
      ↓
  Summarization          ← Pass chunks to LLM (e.g. Llama via Ollama) for final answer
```

## Part 1: Data Mining through NewsData.io API connection

In [ ]:
import requests

url = "https://newsdata.io/api/1/latest"

params = {
    "apikey": "API_key",
    "q": "Artificial Intelligence"
}

response = requests.get(url, params=params)
data = response.json()

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print(f"Total results: {data['totalResults']}")
else:
    print(f"Error {response.status_code}: {response.text}")

Total results: 3487


In [2]:
Append_article = []

for i in range(len(data["results"])):
    article = data["results"][i]
    Append_article.append({
        "Title": article["title"],
        "Description": article["description"]
    })

# Print to verify
for item in Append_article:
    print(f"Title       : {item['Title']}")
    print(f"Description : {item['Description']}")
    print("-" * 60)

Title       : Smaller Public Service May Cost New Zealanders More
Description : Aotearoa New Zealand's government is attempting one of the country's largest public service reforms in decades - and betting artificial intelligence
------------------------------------------------------------
Title       : Apple’s 2026 AI Strategy Shifts Toward On-Device Models
Description : Apple is expanding on-device AI across iPhones, iPads, and Macs as it reduces reliance on cloud-based systems used by rivals like Google and Microsoft. The strategy focuses on privacy, speed, and tighter hardware integration, reflecting Apple’s belief that personal devices could define the next phase of artificial intelligence. The post Apple’s 2026 AI Strategy Shifts Toward On-Device Models appeared first on Memeburn .
------------------------------------------------------------
Title       : Pope Leo to release long-awaited AI manifesto on ethical risks and global impact
Description : Pope Leo XIV will release on Mon

In [3]:
documents = []

for item in Append_article:
    text = f"Title: {item['Title']}\nDescription: {item['Description']}"
    documents.append(text)

print(f"Total documents: {len(documents)}")
print(documents[0])  # preview first article

Total documents: 10
Title: Smaller Public Service May Cost New Zealanders More
Description: Aotearoa New Zealand's government is attempting one of the country's largest public service reforms in decades - and betting artificial intelligence


## Part 2: Chunking
Split each document into smaller chunks for better retrieval:

In [4]:
#!pip install langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

all_chunks = []

for i, doc in enumerate(documents):
    chunks = splitter.split_text(doc)
    for j, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"doc_{i}_chunk_{j}",
            "text": chunk,
            "source_index": i
        })

print(f"Total chunks: {len(all_chunks)}")

c:\Users\Ammar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 20


> **Note:** Since news articles are usually short, a `chunk_size` of `300` with `chunk_overlap` of `50` works well. If descriptions are under 300 characters, each article will be a single chunk — which is fine.

## Part 3: Embedding
Convert each chunk into a vector using a local embedding model:

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts     = [c["text"] for c in all_chunks]
ids       = [c["id"]   for c in all_chunks]
metadatas = [{"source_index": c["source_index"]} for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True).tolist()

print(f"Embeddings: {len(embeddings)} x {len(embeddings[0])}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.92it/s]

Embeddings: 20 x 384


## Part 4: Index into ChromaDB
Store the embedded chunks into a local ChromaDB vector database:

In [7]:
import chromadb

client     = chromadb.PersistentClient(path="./news_chroma_db")
collection = client.get_or_create_collection(name="artificial_intelligence")

collection.add(
    ids        = ids,
    embeddings = embeddings,
    documents  = texts,
    metadatas  = metadatas
)

print(f"Total indexed chunks: {collection.count()}")

Total indexed chunks: 20


## Part 5: Query the Index
Search for the most relevant chunks based on a query:

In [8]:
query        = "AI latest achievements"
query_vector = model.encode([query]).tolist()

results = collection.query(
    query_embeddings = query_vector,
    n_results        = 3
)

for doc, score in zip(results["documents"][0], results["distances"][0]):
    print(f"Score : {score:.4f}")
    print(f"Chunk : {doc}")
    print("-" * 60)

Score : 1.0174
Chunk : Description: Pope Leo XIV will release on Monday his long-awaited manifesto on artificial intelligence (AI), a bid to address ethical and social challenges as the technology rapidly develops worldwide. The Holy Father will attend the presentation of the “Magnifica Humanitas” (Magnificent Humanity)
------------------------------------------------------------
Score : 1.0243
Chunk : Description: We are counting courses completed, certifications issued, and AI tools deployed. What we are not counting — not carefully enough, anyway — is who is being left out of all of it. There is a version of the AI upskilling story that sounds almost aspirational. Governments are investing.
------------------------------------------------------------
Score : 1.0304
Chunk : Description: VATICAN CITY, Holy See — Pope Leo XIV will release on Monday his long-awaited manifesto on artificial intelligence (AI), a bid to address ethical and social challenges as the technology rapidly develo

In [9]:
query        = "AI latest challenges"
query_vector = model.encode([query]).tolist()

results = collection.query(
    query_embeddings = query_vector,
    n_results        = 3
)

for doc, score in zip(results["documents"][0], results["distances"][0]):
    print(f"Score : {score:.4f}")
    print(f"Chunk : {doc}")
    print("-" * 60)

Score : 1.0451
Chunk : Description: VATICAN CITY, Holy See — Pope Leo XIV will release on Monday his long-awaited manifesto on artificial intelligence (AI), a bid to address ethical and social challenges as the technology rapidly develops worldwide. The US pope will attend the presentation of the “Magnifica Humanitas”
------------------------------------------------------------
Score : 1.0497
Chunk : Title: Apple’s 2026 AI Strategy Shifts Toward On-Device Models
------------------------------------------------------------
Score : 1.0510
Chunk : Description: Pope Leo XIV will release on Monday his long-awaited manifesto on artificial intelligence (AI), a bid to address ethical and social challenges as the technology rapidly develops worldwide. The Holy Father will attend the presentation of the “Magnifica Humanitas” (Magnificent Humanity)
------------------------------------------------------------


In [10]:
query        = "AI latest risks"
query_vector = model.encode([query]).tolist()

results = collection.query(
    query_embeddings = query_vector,
    n_results        = 3
)

for doc, score in zip(results["documents"][0], results["distances"][0]):
    print(f"Score : {score:.4f}")
    print(f"Chunk : {doc}")
    print("-" * 60)

Score : 0.8623
Chunk : Title: Pope Leo to release long-awaited AI manifesto on ethical risks and global impact
------------------------------------------------------------
Score : 1.0382
Chunk : Description: We are counting courses completed, certifications issued, and AI tools deployed. What we are not counting — not carefully enough, anyway — is who is being left out of all of it. There is a version of the AI upskilling story that sounds almost aspirational. Governments are investing.
------------------------------------------------------------
Score : 1.0519
Chunk : Title: Apple’s 2026 AI Strategy Shifts Toward On-Device Models
------------------------------------------------------------


> **Note:** ChromaDB uses distance scores — **lower = more similar**.